In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
df = spark.read.option("header", True).csv("/Workspace/Users/11232631_sanjaykumar@mmumullana.org/DeltaLakeAssignment/customer_master.csv")

display(df)

CustomerID,Name,Email,City,Phone
101,Amit Sharma,amit.sharma@gmail.com,Delhi,9876543210
102,Priya Singh,priya.singh@gmail.com,Mumbai,9876543211
103,Rahul Verma,rahul.verma@gmail.com,Bangalore,9876543212
104,Neha Gupta,neha.gupta@gmail.com,Kolkata,9876543213
105,Rohit Kumar,rohit.kumar@gmail.com,Chennai,9876543214
106,Anjali Mehta,anjali.mehta@gmail.com,Pune,9876543215
107,Vikas Yadav,null,Lucknow,9876543216
108,Sneha Joshi,sneha.joshi@gmail.com,null,9876543217
109,Karan Patel,karan.patel@gmail.com,Ahmedabad,9876543218
110,Pooja Kapoor,pooja.kapoor@gmail.com,Jaipur,9876543219


In [0]:
df.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Phone: string (nullable = true)



In [0]:
print(df.count())

21


In [0]:
df = df.fillna({
    "City":"Unknown",
    "Email":"Not Available"
})

In [0]:
df = df.dropDuplicates()

In [0]:
print("Rows:", df.count())

print("Duplicates:",
      df.count() - df.dropDuplicates().count())

Rows: 20
Duplicates: 0


In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_table")

In [0]:
delta_df = spark.table("customer_table")

display(delta_df)

CustomerID,Name,Email,City,Phone
108,Sneha Joshi,sneha.joshi@gmail.com,Unknown,9876543217
102,Priya Singh,priya.singh@gmail.com,Mumbai,9876543211
107,Vikas Yadav,Not Available,Lucknow,9876543216
110,Pooja Kapoor,pooja.kapoor@gmail.com,Jaipur,9876543219
115,Arjun Malhotra,arjun.malhotra@gmail.com,Delhi,9876543224
103,Rahul Verma,rahul.verma@gmail.com,Bangalore,9876543212
101,Amit Sharma,amit.sharma@gmail.com,Delhi,9876543210
106,Anjali Mehta,anjali.mehta@gmail.com,Pune,9876543215
112,Meena Das,meena.das@gmail.com,Bhubaneswar,9876543221
113,Suresh Reddy,suresh.reddy@gmail.com,Hyderabad,9876543222


In [0]:
import os

print(os.getcwd())

/Workspace/Users/11232631_sanjaykumar@mmumullana.org/DeltaLakeAssignment


In [0]:
display(dbutils.fs.ls("/"))

path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


In [0]:
inc_df = spark.read.option("header", True).csv("/Workspace/Users/11232631_sanjaykumar@mmumullana.org/DeltaLakeAssignment/customer_incremental.csv")

display(inc_df)

CustomerID,Name,Email,City,Phone
103,Rahul Verma,rahul.verma@gmail.com,Hyderabad,9999999991
106,Anjali Mehta,anjali.mehta@gmail.com,Bengaluru,9999999992
110,Pooja Kapoor,pooja.kapoor@outlook.com,Jaipur,9999999993
115,Arjun Malhotra,arjun.malhotra@gmail.com,Gurgaon,9999999994
121,Sachin Tiwari,sachin.tiwari@gmail.com,Noida,9999999995
122,Divya Sharma,divya.sharma@gmail.com,Delhi,9999999996
123,Akash Gupta,akash.gupta@gmail.com,Pune,9999999997
124,Nisha Verma,nisha.verma@gmail.com,Mumbai,9999999998
125,Harsh Patel,harsh.patel@gmail.com,Ahmedabad,9999999999


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "customer_table")

In [0]:
(
    deltaTable.alias("target")
    .merge(
        inc_df.alias("source"),
        "target.CustomerID = source.CustomerID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
final_df = spark.table("customer_table")

display(final_df)

CustomerID,Name,Email,City,Phone
108,Sneha Joshi,sneha.joshi@gmail.com,Unknown,9876543217
102,Priya Singh,priya.singh@gmail.com,Mumbai,9876543211
107,Vikas Yadav,Not Available,Lucknow,9876543216
101,Amit Sharma,amit.sharma@gmail.com,Delhi,9876543210
112,Meena Das,meena.das@gmail.com,Bhubaneswar,9876543221
113,Suresh Reddy,suresh.reddy@gmail.com,Hyderabad,9876543222
105,Rohit Kumar,rohit.kumar@gmail.com,Chennai,9876543214
116,Deepa Iyer,deepa.iyer@gmail.com,Chennai,9876543225
119,Manoj Sinha,manoj.sinha@gmail.com,Patna,9876543228
109,Karan Patel,karan.patel@gmail.com,Ahmedabad,9876543218


In [0]:
print("Final Row Count:", final_df.count())

Final Row Count: 25


In [0]:
duplicates = (
    final_df
    .groupBy("CustomerID")
    .count()
    .filter("count > 1")
)

display(duplicates)

CustomerID,count


In [0]:
print("Total Rows:", final_df.count())

print(
    "Distinct Customers:",
    final_df.select("CustomerID").distinct().count()
)

Total Rows: 25
Distinct Customers: 25


In [0]:
display(final_df.orderBy("CustomerID"))

CustomerID,Name,Email,City,Phone
101,Amit Sharma,amit.sharma@gmail.com,Delhi,9876543210
102,Priya Singh,priya.singh@gmail.com,Mumbai,9876543211
103,Rahul Verma,rahul.verma@gmail.com,Hyderabad,9999999991
104,Neha Gupta,neha.gupta@gmail.com,Kolkata,9876543213
105,Rohit Kumar,rohit.kumar@gmail.com,Chennai,9876543214
106,Anjali Mehta,anjali.mehta@gmail.com,Bengaluru,9999999992
107,Vikas Yadav,Not Available,Lucknow,9876543216
108,Sneha Joshi,sneha.joshi@gmail.com,Unknown,9876543217
109,Karan Patel,karan.patel@gmail.com,Ahmedabad,9876543218
110,Pooja Kapoor,pooja.kapoor@outlook.com,Jaipur,9999999993


# Assignment Summary

1. Loaded customer master dataset.
2. Cleaned null values.
3. Removed duplicate records.
4. Stored data in Delta Lake.
5. Loaded incremental customer dataset.
6. Applied Delta MERGE operation.
7. Existing records were updated.
8. New records were inserted.
9. Validated row count and duplicates.
10. Displayed final merged dataset.